# Fine-Tuning an LLM for Structured Job Posting Extraction

## Objective

The goal of this project is to fine-tune a small language model
to extract structured information from unstructured job postings.

The model will extract five categories of information:

1. Job title
2. Required technical skills
3. Minimum years of experience
4. Remote work status
5. Salary range

The extracted information will be returned in a consistent
JSON format.

## Research Question

Does fine-tuning a small language model improve JSON validity
and field-level extraction accuracy compared with the
original model?

## Method

We will use Low-Rank Adaptation (LoRA) to fine-tune
a pretrained language model.

We will then compare the performance of the original
and fine-tuned models on unseen job postings.

## Evaluation Metrics

1. Valid JSON Rate
2. Field-Level Accuracy

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1e9,
            2
        ),
        "GB"
    )
else:
    print("No GPU detected.")

PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU: Tesla T4
GPU Memory: 15.64 GB


In [ ]:
import json

example_posting = """
We are hiring a Data Scientist!

Candidates must have at least 3 years of experience
with Python, SQL, and machine learning.

This is a fully remote position.

Annual salary: $110,000 - $145,000.
"""

expected_output = {
    "title": "Data Scientist",
    "required_skills": ["Python", "SQL", "Machine Learning"],
    "years_experience": 3,
    "remote": True,
    "salary_min": 110000,
    "salary_max": 145000
}

print("JOB POSTING:")
print(example_posting)

print("\nEXPECTED JSON OUTPUT:")
print(json.dumps(expected_output, indent=2))

JOB POSTING:

We are hiring a Data Scientist!

Candidates must have at least 3 years of experience
with Python, SQL, and machine learning.

This is a fully remote position.

Annual salary: $110,000 - $145,000.


EXPECTED JSON OUTPUT:
{
  "title": "Data Scientist",
  "required_skills": [
    "Python",
    "SQL",
    "Machine Learning"
  ],
  "years_experience": 3,
  "remote": true,
  "salary_min": 110000,
  "salary_max": 145000
}


In [ ]:
import pandas as pd
import json
import random

random.seed(42)

# Job roles and their associated technical skills
job_roles = {
    "Data Scientist": [
        "Python", "SQL", "Machine Learning"
    ],
    "Data Analyst": [
        "SQL", "Excel", "Tableau"
    ],
    "Machine Learning Engineer": [
        "Python", "PyTorch", "Docker"
    ],
    "Data Engineer": [
        "Python", "SQL", "Apache Spark"
    ],
    "AI Engineer": [
        "Python", "LLMs", "PyTorch"
    ],
    "Business Intelligence Analyst": [
        "SQL", "Power BI", "Excel"
    ],
    "NLP Engineer": [
        "Python", "NLP", "Transformers"
    ],
    "Cloud Data Engineer": [
        "AWS", "Python", "Spark"
    ],
    "Research Scientist": [
        "Python", "Machine Learning", "Statistics"
    ],
    "Analytics Engineer": [
        "SQL", "dbt", "Python"
    ]
}

# Different ways companies describe work arrangements
work_arrangements = [
    ("This is a fully remote position.", True),
    ("Work from anywhere in the United States.", True),
    ("This role is onsite at our office.", False),
    ("This is a hybrid position requiring office attendance.", False),
    ("Employees must work from our headquarters.", False)
]

# Different job description writing styles
templates = [
    (
        "We are hiring a {title}! "
        "Candidates must have at least {years} years of experience. "
        "Required skills include {skills}. "
        "{work} "
        "Annual salary: ${salary_min:,} - ${salary_max:,}."
    ),
    (
        "Join our team as a {title}. "
        "We're looking for someone with {years}+ years of professional experience. "
        "You should be proficient in {skills}. "
        "{work} "
        "Compensation ranges from ${salary_min:,} to ${salary_max:,} per year."
    ),
    (
        "Exciting opportunity: {title}. "
        "Applicants need a minimum of {years} years of relevant experience. "
        "Technical requirements: {skills}. "
        "{work} "
        "Expected annual pay is ${salary_min:,}–${salary_max:,}."
    ),
    (
        "Our growing team is looking for a {title}. "
        "We require {years} years of experience and knowledge of {skills}. "
        "{work} "
        "The salary range is ${salary_min:,} to ${salary_max:,} annually."
    ),
    (
        "Now hiring: {title}! "
        "Ideal applicants have {years} or more years of experience. "
        "You must be skilled in {skills}. "
        "{work} "
        "We offer annual compensation between ${salary_min:,} and ${salary_max:,}."
    )
]

# Generate 50 labeled examples
records = []

for i in range(50):

    title = random.choice(list(job_roles.keys()))

    skills = job_roles[title]

    years = random.randint(1, 6)

    work_text, remote_status = random.choice(work_arrangements)

    salary_min = random.randrange(60000, 150001, 5000)

    salary_max = salary_min + random.randrange(20000, 50001, 5000)

    template = random.choice(templates)

    posting = template.format(
        title=title,
        years=years,
        skills=", ".join(skills),
        work=work_text,
        salary_min=salary_min,
        salary_max=salary_max
    )

    expected_output = {
        "title": title,
        "required_skills": skills,
        "years_experience": years,
        "remote": remote_status,
        "salary_min": salary_min,
        "salary_max": salary_max
    }

    records.append({
        "id": i + 1,
        "input": posting,
        "output": expected_output
    })

# Create DataFrame
df = pd.DataFrame(records)

print("Total examples:", len(df))

display(df.head())

Total examples: 50


,id,input,output
0,1,Join our team as a Data Analyst. We're looking...,"{'title': 'Data Analyst', 'required_skills': [..."
1,2,Our growing team is looking for a Data Analyst...,"{'title': 'Data Analyst', 'required_skills': [..."
2,3,Now hiring: Data Scientist! Ideal applicants h...,"{'title': 'Data Scientist', 'required_skills':..."
3,4,Now hiring: Analytics Engineer! Ideal applican...,"{'title': 'Analytics Engineer', 'required_skil..."
4,5,We are hiring a NLP Engineer! Candidates must ...,"{'title': 'NLP Engineer', 'required_skills': [..."


In [ ]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst Job Posting:")
print(df.loc[0, "input"])

print("\nExpected JSON:")
print(json.dumps(df.loc[0, "output"], indent=2))

Dataset Shape: (50, 3)

Column Names:
['id', 'input', 'output']

First Job Posting:
Join our team as a Data Analyst. We're looking for someone with 1+ years of professional experience. You should be proficient in SQL, Excel, Tableau. This role is onsite at our office. Compensation ranges from $95,000 to $120,000 per year.

Expected JSON:
{
  "title": "Data Analyst",
  "required_skills": [
    "SQL",
    "Excel",
    "Tableau"
  ],
  "years_experience": 1,
  "remote": false,
  "salary_min": 95000,
  "salary_max": 120000
}


In [ ]:
print("Total Postings:", len(df))

print(
    "Duplicate Postings:",
    df["input"].duplicated().sum()
)

print(
    "Unique Job Titles:",
    df["output"].apply(lambda x: x["title"]).nunique()
)

print("\nMissing Values:")
print(df.isnull().sum())

Total Postings: 50
Duplicate Postings: 0
Unique Job Titles: 10

Missing Values:
id        0
input     0
output    0
dtype: int64


In [ ]:
# More realistic training examples

challenging_examples = [
    {
        "input": (
            "We're hiring a Data Scientist. Python, SQL, and "
            "machine learning are required. AWS is a plus. "
            "Applicants need at least 3 years of experience. "
            "This is a fully remote opportunity. "
            "Salary will be discussed during interviews."
        ),
        "output": {
            "title": "Data Scientist",
            "required_skills": ["Python", "SQL", "Machine Learning"],
            "years_experience": 3,
            "remote": True,
            "salary_min": None,
            "salary_max": None
        }
    },
    {
        "input": (
            "Looking for an entry-level Data Analyst. "
            "Candidates must know SQL and Excel. "
            "Experience with Tableau would be beneficial. "
            "This position is onsite in Seattle. "
            "Annual compensation is $65k-$80k."
        ),
        "output": {
            "title": "Data Analyst",
            "required_skills": ["SQL", "Excel"],
            "years_experience": None,
            "remote": False,
            "salary_min": 65000,
            "salary_max": 80000
        }
    },
    {
        "input": (
            "Machine Learning Engineer needed! "
            "We require Python, PyTorch, and Docker skills. "
            "Minimum 4 years of professional experience. "
            "The role is hybrid, with three office days per week. "
            "Pay ranges from $120,000 to $155,000 annually."
        ),
        "output": {
            "title": "Machine Learning Engineer",
            "required_skills": ["Python", "PyTorch", "Docker"],
            "years_experience": 4,
            "remote": False,
            "salary_min": 120000,
            "salary_max": 155000
        }
    },
    {
        "input": (
            "Our team is searching for a Data Engineer. "
            "Strong Python, SQL, and Apache Spark skills are essential. "
            "AWS knowledge is preferred. "
            "Work from anywhere in the United States. "
            "Compensation: $105K-$135K per year."
        ),
        "output": {
            "title": "Data Engineer",
            "required_skills": ["Python", "SQL", "Apache Spark"],
            "years_experience": None,
            "remote": True,
            "salary_min": 105000,
            "salary_max": 135000
        }
    },
    {
        "input": (
            "AI Engineer opening! "
            "Applicants must have experience with Python, "
            "LLMs, and PyTorch. "
            "At least 2 years of relevant work experience required. "
            "Position based in San Francisco; onsite attendance required. "
            "Salary information is not provided."
        ),
        "output": {
            "title": "AI Engineer",
            "required_skills": ["Python", "LLMs", "PyTorch"],
            "years_experience": 2,
            "remote": False,
            "salary_min": None,
            "salary_max": None
        }
    },
    {
        "input": (
            "We are seeking a Business Intelligence Analyst. "
            "SQL and Power BI proficiency are mandatory. "
            "Excel experience is preferred. "
            "Candidates should have 2+ years of relevant experience. "
            "This is a remote-first role with no required office days. "
            "Annual salary: $80,000-$100,000."
        ),
        "output": {
            "title": "Business Intelligence Analyst",
            "required_skills": ["SQL", "Power BI"],
            "years_experience": 2,
            "remote": True,
            "salary_min": 80000,
            "salary_max": 100000
        }
    },
    {
        "input": (
            "NLP Engineer wanted. "
            "Python, natural language processing, and Transformers "
            "are required skills. "
            "Previous industry experience is welcome but not mandatory. "
            "This is a hybrid opportunity in New York. "
            "Annual pay ranges from $115K to $150K."
        ),
        "output": {
            "title": "NLP Engineer",
            "required_skills": ["Python", "NLP", "Transformers"],
            "years_experience": None,
            "remote": False,
            "salary_min": 115000,
            "salary_max": 150000
        }
    },
    {
        "input": (
            "Cloud Data Engineer opportunity. "
            "Candidates must be proficient in AWS, Python, and Spark. "
            "Minimum experience: 5 years. "
            "Employees can work remotely from anywhere in the US. "
            "Salary range is not disclosed."
        ),
        "output": {
            "title": "Cloud Data Engineer",
            "required_skills": ["AWS", "Python", "Spark"],
            "years_experience": 5,
            "remote": True,
            "salary_min": None,
            "salary_max": None
        }
    },
    {
        "input": (
            "Research Scientist position available. "
            "Machine learning and statistics expertise required. "
            "Python programming is also essential. "
            "A PhD is preferred but not required. "
            "We offer $130k-$175k annually. "
            "The position requires working onsite."
        ),
        "output": {
            "title": "Research Scientist",
            "required_skills": ["Python", "Machine Learning", "Statistics"],
            "years_experience": None,
            "remote": False,
            "salary_min": 130000,
            "salary_max": 175000
        }
    },
    {
        "input": (
            "Hiring an Analytics Engineer to support our data team. "
            "Required technical skills include SQL, dbt, and Python. "
            "At least 3 years of analytics experience needed. "
            "Salary: $95,000-$125,000 per year. "
            "Work arrangement will be discussed with selected candidates."
        ),
        "output": {
            "title": "Analytics Engineer",
            "required_skills": ["SQL", "dbt", "Python"],
            "years_experience": 3,
            "remote": None,
            "salary_min": 95000,
            "salary_max": 125000
        }
    }
]

# Replace the first 10 records
for i, example in enumerate(challenging_examples):
    df.at[i, "input"] = example["input"]
    df.at[i, "output"] = example["output"]

print("Updated dataset size:", len(df))

print("\nExample with missing information:")
print(df.loc[0, "input"])

print("\nExpected output:")
print(json.dumps(df.loc[0, "output"], indent=2))

Updated dataset size: 50

Example with missing information:
We're hiring a Data Scientist. Python, SQL, and machine learning are required. AWS is a plus. Applicants need at least 3 years of experience. This is a fully remote opportunity. Salary will be discussed during interviews.

Expected output:
{
  "title": "Data Scientist",
  "required_skills": [
    "Python",
    "SQL",
    "Machine Learning"
  ],
  "years_experience": 3,
  "remote": true,
  "salary_min": null,
  "salary_max": null
}


In [ ]:
print("Total Examples:", len(df))

print(
    "Duplicate Postings:",
    df["input"].duplicated().sum()
)

# Count missing values within expected outputs
fields = [
    "title",
    "required_skills",
    "years_experience",
    "remote",
    "salary_min",
    "salary_max"
]

for field in fields:
    missing_count = sum(
        record[field] is None
        for record in df["output"]
    )

    print(f"{field}: {missing_count} missing")

Total Examples: 50
Duplicate Postings: 0
title: 0 missing
required_skills: 0 missing
years_experience: 4 missing
remote: 1 missing
salary_min: 3 missing
salary_max: 3 missing


In [ ]:
# Keep the first 40 examples for training
train_df = df.iloc[:40].copy().reset_index(drop=True)

print("Training examples:", len(train_df))

display(train_df.head(3))

Training examples: 40


,id,input,output
0,1,"We're hiring a Data Scientist. Python, SQL, an...","{'title': 'Data Scientist', 'required_skills':..."
1,2,Looking for an entry-level Data Analyst. Candi...,"{'title': 'Data Analyst', 'required_skills': [..."
2,3,Machine Learning Engineer needed! We require P...,"{'title': 'Machine Learning Engineer', 'requir..."


In [ ]:
test_examples = [
    {
        "input": (
            "Position: Data Scientist\n"
            "Location: Remote - United States\n"
            "Our analytics division is expanding. We need someone "
            "comfortable working with Python, SQL, and statistical modeling. "
            "Candidates must have no fewer than 4 years of relevant experience. "
            "Familiarity with AWS is a bonus. "
            "The annual compensation package ranges between $125K and $160K."
        ),
        "output": {
            "title": "Data Scientist",
            "required_skills": ["Python", "SQL", "Statistics"],
            "years_experience": 4,
            "remote": True,
            "salary_min": 125000,
            "salary_max": 160000
        }
    },
    {
        "input": (
            "Junior Data Analyst | Seattle, WA\n"
            "We're expanding our reporting team. "
            "You should know Excel and SQL. "
            "Tableau would be nice to have, but isn't mandatory. "
            "This role requires working from our Seattle office. "
            "Starting pay is $72,000 annually. "
            "No previous professional experience is required."
        ),
        "output": {
            "title": "Junior Data Analyst",
            "required_skills": ["Excel", "SQL"],
            "years_experience": 0,
            "remote": False,
            "salary_min": 72000,
            "salary_max": 72000
        }
    },
    {
        "input": (
            "Machine Learning Engineer\n"
            "We're searching for someone who can develop and deploy "
            "production ML systems. Python, TensorFlow, and Kubernetes "
            "are mandatory. A background in AWS is preferred. "
            "You should bring at least five years of industry experience. "
            "The team meets in our office twice a week. "
            "Salary: $145,000-$185,000 per year."
        ),
        "output": {
            "title": "Machine Learning Engineer",
            "required_skills": [
                "Python", "TensorFlow", "Kubernetes"
            ],
            "years_experience": 5,
            "remote": False,
            "salary_min": 145000,
            "salary_max": 185000
        }
    },
    {
        "input": (
            "We're looking for a Data Engineer to support "
            "our growing data infrastructure. "
            "Experience with Apache Spark, Python, and SQL is essential. "
            "Knowledge of Airflow is a plus. "
            "The position allows employees to work from anywhere. "
            "Compensation will depend on qualifications."
        ),
        "output": {
            "title": "Data Engineer",
            "required_skills": [
                "Apache Spark", "Python", "SQL"
            ],
            "years_experience": None,
            "remote": True,
            "salary_min": None,
            "salary_max": None
        }
    },
    {
        "input": (
            "AI Engineer — San Jose, California\n"
            "We're building applications powered by large language models. "
            "The successful candidate must understand Python, "
            "LLMs, and vector databases. "
            "At least 2 years of software development experience required. "
            "Onsite work is mandatory. "
            "We offer between $110k and $150k annually."
        ),
        "output": {
            "title": "AI Engineer",
            "required_skills": [
                "Python", "LLMs", "Vector Databases"
            ],
            "years_experience": 2,
            "remote": False,
            "salary_min": 110000,
            "salary_max": 150000
        }
    },
    {
        "input": (
            "Business Intelligence Analyst\n"
            "Help transform business data into actionable insights. "
            "Strong SQL and Power BI skills are required. "
            "Knowledge of Python is desirable but optional. "
            "Candidates need 3+ years of analytical experience. "
            "Annual salary falls between $85K and $110K. "
            "Work location arrangements have not been finalized."
        ),
        "output": {
            "title": "Business Intelligence Analyst",
            "required_skills": ["SQL", "Power BI"],
            "years_experience": 3,
            "remote": None,
            "salary_min": 85000,
            "salary_max": 110000
        }
    },
    {
        "input": (
            "NLP Engineer Opportunity\n"
            "Our research team develops language understanding systems. "
            "Applicants must demonstrate proficiency in Python, "
            "Hugging Face Transformers, and NLP. "
            "We welcome candidates from academic and industry backgrounds. "
            "This opportunity is fully remote. "
            "The expected annual compensation is $120,000 to $155,000."
        ),
        "output": {
            "title": "NLP Engineer",
            "required_skills": [
                "Python", "Hugging Face Transformers", "NLP"
            ],
            "years_experience": None,
            "remote": True,
            "salary_min": 120000,
            "salary_max": 155000
        }
    },
    {
        "input": (
            "Cloud Data Engineer — Austin, Texas\n"
            "Join our cloud infrastructure team. "
            "AWS, Python, and Apache Airflow experience are required. "
            "The ideal candidate has at least 4 years of experience. "
            "Employees must report to our Austin office Monday through Friday. "
            "Salary details are available upon request."
        ),
        "output": {
            "title": "Cloud Data Engineer",
            "required_skills": [
                "AWS", "Python", "Apache Airflow"
            ],
            "years_experience": 4,
            "remote": False,
            "salary_min": None,
            "salary_max": None
        }
    },
    {
        "input": (
            "Research Scientist, Applied Machine Learning\n"
            "We're recruiting a researcher to investigate new ML methods. "
            "Required expertise includes Python, PyTorch, and statistics. "
            "Experience publishing research papers is preferred. "
            "Applicants should have 2 or more years of relevant experience. "
            "The position is remote within the United States. "
            "Base salary is between $135K and $180K annually."
        ),
        "output": {
            "title": "Research Scientist",
            "required_skills": [
                "Python", "PyTorch", "Statistics"
            ],
            "years_experience": 2,
            "remote": True,
            "salary_min": 135000,
            "salary_max": 180000
        }
    },
    {
        "input": (
            "Analytics Engineer\n"
            "We're hiring someone to maintain our analytics pipelines. "
            "SQL and dbt are essential. "
            "Python familiarity is helpful but not required. "
            "This is a hybrid role based in Chicago. "
            "Compensation: $100,000-$130,000 annually. "
            "Prior professional experience is welcome but not mandatory."
        ),
        "output": {
            "title": "Analytics Engineer",
            "required_skills": ["SQL", "dbt"],
            "years_experience": None,
            "remote": False,
            "salary_min": 100000,
            "salary_max": 130000
        }
    }
]

# Create test DataFrame
test_df = pd.DataFrame(test_examples)

# Assign test IDs
test_df.insert(0, "id", range(41, 51))

print("Testing examples:", len(test_df))

display(test_df.head(3))

Testing examples: 10


,id,input,output
0,41,Position: Data Scientist\nLocation: Remote - U...,"{'title': 'Data Scientist', 'required_skills':..."
1,42,"Junior Data Analyst | Seattle, WA\nWe're expan...","{'title': 'Junior Data Analyst', 'required_ski..."
2,43,Machine Learning Engineer\nWe're searching for...,"{'title': 'Machine Learning Engineer', 'requir..."


In [ ]:
# Verify dataset sizes
print("Training examples:", len(train_df))
print("Testing examples:", len(test_df))

# Check for duplicates between datasets
train_postings = set(train_df["input"])
test_postings = set(test_df["input"])

overlap = train_postings.intersection(test_postings)

print("Overlapping postings:", len(overlap))

# Verify all records use the expected output fields
expected_fields = {
    "title",
    "required_skills",
    "years_experience",
    "remote",
    "salary_min",
    "salary_max"
}

all_outputs = list(train_df["output"]) + list(test_df["output"])

schema_correct = all(
    set(output.keys()) == expected_fields
    for output in all_outputs
)

print("Consistent JSON Schema:", schema_correct)

assert len(train_df) == 40
assert len(test_df) == 10
assert len(overlap) == 0
assert schema_correct

print("\nDataset validation passed!")

Training examples: 40
Testing examples: 10
Overlapping postings: 0
Consistent JSON Schema: True

Dataset validation passed!


In [ ]:
# Save training dataset
train_df.to_json(
    "job_postings_train.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

# Save testing dataset
test_df.to_json(
    "job_postings_test.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Training and testing datasets saved successfully!")

Training and testing datasets saved successfully!


In [ ]:
from google.colab import files

files.download("job_postings_train.jsonl")
files.download("job_postings_test.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Test example 41 explicitly requires statistical modeling
test_df.at[0, "output"]["required_skills"] = [
    "Python", "SQL", "Statistical Modeling"
]

# Test example 49 specifies a more detailed job title
test_df.at[8, "output"]["title"] = (
    "Research Scientist, Applied Machine Learning"
)

# Save the corrected test dataset
test_df.to_json(
    "job_postings_test.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Test labels updated successfully!")

Test labels updated successfully!


## 6. Loading the Pretrained Language Model

We will use Qwen2.5-0.5B-Instruct as our base model.

The model will first be evaluated without fine-tuning
to establish baseline performance.

We will then fine-tune the same model using LoRA
and compare its performance on the same test dataset.

In [ ]:
!pip -q install -U transformers accelerate peft datasets

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

# Model identifier
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

# Load pretrained model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")

print("Model:", model_name)

print(
    "Number of parameters:",
    sum(p.numel() for p in model.parameters())
)

print(
    "Model device:",
    next(model.parameters()).device
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct
Number of parameters: 494032768
Model device: cuda:0


## 7. Base Model Evaluation

Before fine-tuning, we will evaluate the original
Qwen2.5-0.5B-Instruct model on 10 unseen job postings.

The model will be instructed to extract job information
into a structured JSON format.

We will save its predictions and use them as our
baseline for comparison with the fine-tuned model.

In [ ]:
# Instructions for the original model

system_prompt = """
You are an information extraction assistant.

Read the job posting and extract the following information:

1. title: The job title
2. required_skills: A list of explicitly required technical skills
3. years_experience: Minimum required years of experience
4. remote: Whether the position is fully remote
5. salary_min: Minimum annual salary in USD
6. salary_max: Maximum annual salary in USD

Rules:
- Return only a valid JSON object.
- Do not include explanations or markdown.
- Use exactly the six specified keys.
- Use null when information is not provided.
- Include only required skills, not preferred skills.
- Use true for fully remote positions.
- Use false for onsite or hybrid positions.
- Use null if remote status is unspecified.
- Convert salary values such as $120k to 120000.
- If a single annual salary is given, use it for both salary fields.
- If no experience is explicitly required, use 0.
- If the experience requirement is not specified, use null.
"""

print("System prompt created successfully!")

System prompt created successfully!


In [ ]:
def extract_job_info(posting, model, tokenizer):

    # Create conversation
    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": posting
        }
    ]

    # Apply the model's chat template
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    # Move inputs to the same device as the model
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # Generate response
    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Identify the generated tokens
    input_length = inputs["input_ids"].shape[-1]

    generated_tokens = outputs[0][input_length:]

    # Decode generated response
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


print("Extraction function updated successfully!")

Extraction function updated successfully!


In [ ]:
sample_posting = test_df.loc[0, "input"]

expected = test_df.loc[0, "output"]

prediction = extract_job_info(
    sample_posting,
    model,
    tokenizer
)

print("ORIGINAL JOB POSTING:")
print(sample_posting)

print("\n" + "=" * 60)

print("\nEXPECTED OUTPUT:")
print(json.dumps(expected, indent=2))

print("\n" + "=" * 60)

print("\nBASE MODEL PREDICTION:")
print(prediction)

ORIGINAL JOB POSTING:
Position: Data Scientist
Location: Remote - United States
Our analytics division is expanding. We need someone comfortable working with Python, SQL, and statistical modeling. Candidates must have no fewer than 4 years of relevant experience. Familiarity with AWS is a bonus. The annual compensation package ranges between $125K and $160K.


EXPECTED OUTPUT:
{
  "title": "Data Scientist",
  "required_skills": [
    "Python",
    "SQL",
    "Statistical Modeling"
  ],
  "years_experience": 4,
  "remote": true,
  "salary_min": 125000,
  "salary_max": 160000
}


BASE MODEL PREDICTION:
```json
{
  "title": "Data Scientist",
  "required_skills": ["Python", "SQL", "Statistical Modeling"],
  "years_experience": null,
  "remote": true,
  "salary_min": 125000,
  "salary_max": 160000
}
```


### 7.4 Base Model Predictions

The original model will generate predictions for all
10 unseen test job postings.

These predictions will be saved before fine-tuning.

We will evaluate JSON validity and field-level accuracy
using the expected outputs in our test dataset.

In [ ]:
from tqdm.auto import tqdm

# Store the original model's predictions
base_predictions = []

print("Generating base model predictions...\n")

for i, row in tqdm(
    test_df.iterrows(),
    total=len(test_df)
):

    posting = row["input"]

    prediction = extract_job_info(
        posting,
        model,
        tokenizer
    )

    base_predictions.append({
        "id": row["id"],
        "input": posting,
        "expected": row["output"],
        "prediction": prediction
    })

print("\nBase model evaluation completed!")

print("Total predictions:", len(base_predictions))

Generating base model predictions...



  0%|          | 0/10 [00:00<?, ?it/s]


Base model evaluation completed!
Total predictions: 10


In [ ]:
import json

def is_valid_json(text):

    try:
        parsed = json.loads(text)
        return isinstance(parsed, dict)

    except (json.JSONDecodeError, TypeError):
        return False


# Check all predictions
json_results = [
    is_valid_json(item["prediction"])
    for item in base_predictions
]

valid_count = sum(json_results)

total_count = len(json_results)

valid_json_rate = (
    valid_count / total_count
) * 100

print("BASE MODEL JSON VALIDITY")
print("-" * 40)

print("Total Predictions:", total_count)

print("Valid JSON Outputs:", valid_count)

print("Invalid JSON Outputs:", total_count - valid_count)

print(
    "Valid JSON Rate:",
    round(valid_json_rate, 2),
    "%"
)

BASE MODEL JSON VALIDITY
----------------------------------------
Total Predictions: 10
Valid JSON Outputs: 0
Invalid JSON Outputs: 10
Valid JSON Rate: 0.0 %


In [ ]:
fields = [
    "title",
    "required_skills",
    "years_experience",
    "remote",
    "salary_min",
    "salary_max"
]


def compare_field(expected, predicted, field):

    # Compare skill lists without considering order
    if field == "required_skills":

        if not isinstance(expected, list):
            return False

        if not isinstance(predicted, list):
            return False

        expected_skills = {
            str(skill).strip().lower()
            for skill in expected
        }

        predicted_skills = {
            str(skill).strip().lower()
            for skill in predicted
        }

        return expected_skills == predicted_skills

    # Compare all other fields
    return (
        type(expected) is type(predicted)
        and expected == predicted
    )


correct_fields = 0
total_fields = len(base_predictions) * len(fields)

for item in base_predictions:

    try:
        predicted = json.loads(item["prediction"])

        if not isinstance(predicted, dict):
            continue

    except (json.JSONDecodeError, TypeError):
        continue

    expected = item["expected"]

    for field in fields:

        if field not in predicted:
            continue

        if compare_field(
            expected[field],
            predicted[field],
            field
        ):
            correct_fields += 1


field_accuracy = (
    correct_fields / total_fields
) * 100

print("BASE MODEL FIELD-LEVEL ACCURACY")
print("-" * 40)

print("Correct Fields:", correct_fields)

print("Total Fields:", total_fields)

print(
    "Field-Level Accuracy:",
    round(field_accuracy, 2),
    "%"
)

BASE MODEL FIELD-LEVEL ACCURACY
----------------------------------------
Correct Fields: 0
Total Fields: 60
Field-Level Accuracy: 0.0 %


In [ ]:
# Convert predictions to DataFrame
base_results_df = pd.DataFrame(base_predictions)

# Save results
base_results_df.to_json(
    "base_model_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Base model predictions saved!")

Base model predictions saved!


In [ ]:
from google.colab import files

files.download("base_model_predictions.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. LoRA Fine-Tuning

We will fine-tune Qwen2.5-0.5B-Instruct using 40 labeled
job postings.

Each training example contains:
- System instructions defining the extraction task
- A job posting as the user input
- The correct JSON as the assistant response

We will use LoRA to adapt the pretrained model
without updating all its parameters.

In [ ]:
%pip install --force-reinstall --no-cache-dir pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 113.7 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 26.0.1
    Uninstalling pyarrow-26.0.1:
      Successfully uninstalled pyarrow-26.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [ ]:
import pandas as pd
import json

from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

train_df = pd.read_json(
    "job_postings_train (1).jsonl",
    lines=True
)

print("Training dataset restored!")
print("Training examples:", len(train_df))

display(train_df.head(3))

In [ ]:
import pyarrow
from datasets import Dataset

print("PyArrow version:", pyarrow.__version__)
print("Hugging Face Datasets imported successfully!")

In [ ]:
import json
from datasets import Dataset

training_conversations = []

for _, row in train_df.iterrows():

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": row["input"]},
        {
            "role": "assistant",
            "content": json.dumps(
                row["output"],
                ensure_ascii=False
            )
        }
    ]

    training_conversations.append({
        "messages": messages
    })

train_dataset = Dataset.from_list(training_conversations)

print("Training examples:", len(train_dataset))
print("Dataset columns:", train_dataset.column_names)
print("Training dataset created successfully!")

Training examples: 40
Dataset columns: ['messages']
Training dataset created successfully!


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

# Load pretrained model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

# Disable caching during training
model.config.use_cache = False

print("Model loaded successfully!")

print("Model:", model_name)

print(
    "Device:",
    next(model.parameters()).device
)

print(
    "Total parameters:",
    sum(p.numel() for p in model.parameters())
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0
Total parameters: 494032768


In [ ]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "v_proj"
    ]
)

# Apply LoRA to the pretrained model
model = get_peft_model(
    model,
    lora_config
)

print("LoRA configuration applied successfully!")

# Show trainable parameters
model.print_trainable_parameters()

LoRA configuration applied successfully!
trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [ ]:
import torch
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

# Train on the assistant's JSON response, not on the instructions/job description.
def tokenize_conversation(example):
    messages = example["messages"]
    prefix_text = tokenizer.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    prefix = tokenizer.encode(prefix_text, add_special_tokens=False)
    full = tokenizer.encode(full_text, add_special_tokens=False)
    response_length = len(tokenizer.encode(messages[-1]["content"], add_special_tokens=False)) + 1
    response_length = min(response_length, len(full))
    labels = [-100] * (len(full) - response_length) + full[-response_length:]
    return {
        "input_ids": full[:1024],
        "attention_mask": [1] * min(len(full), 1024),
        "labels": labels[:1024]
    }

tokenized_train = train_dataset.map(tokenize_conversation, remove_columns=["messages"])
assert len(tokenized_train) == 40
assert all(any(label != -100 for label in row["labels"]) for row in tokenized_train)

training_args = TrainingArguments(
    output_dir="/content/job_posting_lora_training",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100),
)
training_result = trainer.train()
print("Final training loss:", training_result.training_loss)

adapter_dir = "/content/job_posting_lora_adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Saved LoRA adapter to:", adapter_dir)

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Step,Training Loss
5,0.049361
10,0.021544
15,0.017971
20,0.022179
25,0.015490
30,0.012690


Final training loss: 0.023205942163864773


Saved LoRA adapter to: /content/job_posting_lora_adapter


In [ ]:
required = ["model", "tokenizer", "train_dataset", "system_prompt"]

for name in required:
    print(f"{name}: {'Loaded' if name in globals() else 'Missing'}")

model: Missing
tokenizer: Missing
train_dataset: Missing
system_prompt: Missing


In [ ]:
import json
import re
import pandas as pd

model.eval()
fields = ["title", "required_skills", "years_experience", "remote", "salary_min", "salary_max"]
results = []

for _, row in test_df.iterrows():
    conversation = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": row["input"]},
    ]
    prompt = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=180, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw, flags=re.IGNORECASE).strip()
    try:
        prediction = json.loads(cleaned)
        valid_json = isinstance(prediction, dict)
    except (json.JSONDecodeError, TypeError):
        prediction = {}
        valid_json = False
    expected = row["output"]
    field_matches = {}
    for field in fields:
        predicted_value = prediction.get(field)
        expected_value = expected.get(field)
        if field == "required_skills":
            match = isinstance(predicted_value, list) and sorted(str(x).lower() for x in predicted_value) == sorted(str(x).lower() for x in expected_value)
        else:
            match = predicted_value == expected_value
        field_matches[field] = match
    results.append({"id": row.get("id", len(results)+41), "raw": raw, "valid_json": valid_json, "matches": field_matches, "prediction": prediction, "expected": expected})

fine_tuned_json_validity = sum(r["valid_json"] for r in results) / len(results)
fine_tuned_field_accuracy = sum(sum(r["matches"].values()) for r in results) / (len(results) * len(fields))
print(f"Fine-tuned JSON validity: {fine_tuned_json_validity:.1%}")
print(f"Fine-tuned field-level accuracy: {fine_tuned_field_accuracy:.1%}")
print("Per-field correct counts:", {f:sum(r["matches"][f] for r in results) for f in fields})
display(pd.DataFrame([{"id":r["id"],"valid_json":r["valid_json"],"correct_fields":sum(r["matches"].values()),"output":r["raw"][:130]} for r in results]))

with open("/content/fine_tuned_predictions.jsonl", "w") as f:
    for result in results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")
print("Saved predictions to /content/fine_tuned_predictions.jsonl")

Fine-tuned JSON validity: 100.0%
Fine-tuned field-level accuracy: 83.3%
Per-field correct counts: {'title': 9, 'required_skills': 8, 'years_experience': 8, 'remote': 9, 'salary_min': 7, 'salary_max': 9}


,id,valid_json,correct_fields,output
0,41,True,6,"```json\n{\n ""title"": ""Data Scientist"",\n ""r..."
1,42,True,4,"```json\n{\n ""title"": ""Junior Data Analyst"",\..."
2,43,True,6,"```json\n{\n ""title"": ""Machine Learning Engin..."
3,44,True,5,"```json\n{\n ""title"": ""Data Engineer"",\n ""re..."
4,45,True,5,"```json\n{\n ""title"": ""AI Engineer"",\n ""requ..."
5,46,True,5,"{\n ""title"": ""Business Intelligence Analyst"",..."
6,47,True,5,"```json\n{\n ""title"": ""NLP Engineer"",\n ""req..."
7,48,True,5,"```json\n{\n ""title"": ""Cloud Data Engineer"",\..."
8,49,True,5,"```json\n{\n ""title"": ""Research Scientist"",\n..."
9,50,True,4,"```json\n{\n ""title"": ""Analytics Engineer"",\n..."


Saved predictions to /content/fine_tuned_predictions.jsonl


In [ ]:
# Strict JSON means json.loads accepts the raw model response without removing code fences.
raw_valid = 0
for result in results:
    try:
        raw_valid += isinstance(json.loads(result["raw"]), dict)
    except (json.JSONDecodeError, TypeError):
        pass
print(f"Strict raw JSON validity: {raw_valid}/{len(results)} ({raw_valid/len(results):.1%})")
print(f"JSON validity after removing optional markdown fences: {sum(r['valid_json'] for r in results)}/{len(results)}")

Strict raw JSON validity: 1/10 (10.0%)
JSON validity after removing optional markdown fences: 10/10


In [ ]:
import shutil
from google.colab import files

# Bundle adapter weights, tokenizer, and held-out predictions for future use.
shutil.copy("/content/fine_tuned_predictions.jsonl", "/content/job_posting_lora_adapter/fine_tuned_predictions.jsonl")
archive_path = shutil.make_archive("/content/job_posting_lora_adapter", "zip", "/content/job_posting_lora_adapter")
print("Export ready:", archive_path)
files.download(archive_path)

Export ready: /content/job_posting_lora_adapter.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>